# Olist E-Commerce - 01 : Understanding the Data

**Author:** Diego Ospina | **Dataset:** Brazilian E-Commerce Public Dataset by Olist

> This is the first notebook of a portfolio project. The purpose here is to **read every raw file and understand its structure, granularity, and data quality** before any cleaning or modeling. Comments are in Spanish; titles and narrative are in English.

### What we will do
1. Load the 9 raw CSV files.
2. Build a data dictionary (columns, types, shapes).
3. Assess data quality: missing values, duplicates, key uniqueness.
4. Validate referential integrity between tables.
5. Look at key categorical distributions.

### Repository layout
- `data/raw/` - original CSV files (read-only, never modified).
- `data/processed/` - cleaned datasets produced by `02_cleaning_preprocessing.ipynb`.
- `images/`, `reports/` - figures and reports generated downstream.

## 1. Setup

We configure the display and verify the library versions we are using.

In [1]:
import pandas as pd
import numpy as np
import os

print('pandas', pd.__version__)
print('numpy', np.__version__)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW = os.path.join('..', 'data', 'raw')
PROCESSED = os.path.join('..', 'data', 'processed')
IMAGES = os.path.join('..', 'images')
REPORTS = os.path.join('..', 'reports')
for d in (PROCESSED, IMAGES, REPORTS):
    os.makedirs(d, exist_ok=True)
print('output dirs ready')

pandas 2.3.3
numpy 2.2.6
output dirs ready


## 2. Load the raw datasets

Each file corresponds to a logical table of the Olist marketplace. We load them all and note the **reason for each file**:

| File | Role |
|---|---|
| `olist_customers_dataset.csv` | Customers and their delivery location |
| `olist_orders_dataset.csv` | Orders with all status timestamps |
| `olist_order_items_dataset.csv` | Each product/seller line inside an order |
| `olist_order_payments_dataset.csv` | Payment methods per order |
| `olist_order_reviews_dataset.csv` | Customer reviews per order |
| `olist_products_dataset.csv` | Products catalog and physical attributes |
| `olist_sellers_dataset.csv` | Sellers and their location |
| `olist_geolocation_dataset.csv` | Zip-code to lat/lng mapping |
| `product_category_name_translation.csv` | Portuguese to English category names |

In [2]:
# Lectura de las tablas crudas. Nota: los prefijos de codigo postal se leen como string
# para evitar perder ceros a la izquierda.
customers  = pd.read_csv(os.path.join(RAW, 'olist_customers_dataset.csv'), dtype={'customer_zip_code_prefix': str})
geolocation= pd.read_csv(os.path.join(RAW, 'olist_geolocation_dataset.csv'), dtype={'geolocation_zip_code_prefix': str})
order_items= pd.read_csv(os.path.join(RAW, 'olist_order_items_dataset.csv'))
payments   = pd.read_csv(os.path.join(RAW, 'olist_order_payments_dataset.csv'))
reviews    = pd.read_csv(os.path.join(RAW, 'olist_order_reviews_dataset.csv'))
orders     = pd.read_csv(os.path.join(RAW, 'olist_orders_dataset.csv'))
products   = pd.read_csv(os.path.join(RAW, 'olist_products_dataset.csv'))
sellers    = pd.read_csv(os.path.join(RAW, 'olist_sellers_dataset.csv'), dtype={'seller_zip_code_prefix': str})
categories = pd.read_csv(os.path.join(RAW, 'product_category_name_translation.csv'))

raw = {'customers': customers, 'orders': orders, 'order_items': order_items, 'products': products, 'payments': payments, 'reviews': reviews, 'sellers': sellers, 'geolocation': geolocation, 'categories': categories}

## 3. Dataset inventory

We print the number of rows/columns of each table to confirm the size and scale of the data.

In [3]:
for name, df in raw.items():
    print(f"{name:14s} -> {df.shape[0]:>9,} rows x {df.shape[1]} cols")

total_mb = sum(df.memory_usage(deep=True).sum() for df in raw.values()) / 1e6
print()
print('Total raw memory (MB):', round(total_mb, 1))

customers      ->    99,441 rows x 5 cols
orders         ->    99,441 rows x 8 cols
order_items    ->   112,650 rows x 7 cols
products       ->    32,951 rows x 9 cols
payments       ->   103,886 rows x 5 cols
reviews        ->    99,224 rows x 7 cols
sellers        ->     3,095 rows x 4 cols
geolocation    -> 1,000,163 rows x 5 cols
categories     ->        71 rows x 2 cols



Total raw memory (MB): 417.4


### 3.1 Granularity

It is critical to know what **one row** means in each table:

- `orders`: one row per `order_id` (an order).
- `order_items`: one row per `(order_id, order_item_id)` - an order with several products has several rows.
- `payments`: one row per `(order_id, payment_sequential)` - an order can be paid in installments or with multiple methods.
- `reviews`: one row per `(order_id, review_id)` - an order can have more than one review.

Let us confirm the cardinalities with pandas.

In [4]:
def cardinality(df, cols):
    # Devuelve n de ids unicos vs filas totales para entender la granularidad
    return {c: f'{df[c].nunique():,} / {len(df):,}' for c in cols}

print('orders       ', cardinality(order_items, ['order_id']))
print('order_items  ', cardinality(order_items, ['order_id', 'order_item_id']))
print('payments     ', cardinality(payments, ['order_id', 'payment_sequential']))
print('reviews      ', cardinality(reviews, ['order_id', 'review_id']))
print('customers    ', cardinality(customers, ['customer_id', 'customer_unique_id']))
print('products     ', cardinality(products, ['product_id']))
print('sellers      ', cardinality(sellers, ['seller_id']))

orders        {'order_id': '98,666 / 112,650'}
order_items   {'order_id': '98,666 / 112,650', 'order_item_id': '21 / 112,650'}
payments      {'order_id': '99,440 / 103,886', 'payment_sequential': '29 / 103,886'}
reviews       {'order_id': '98,673 / 99,224', 'review_id': '98,410 / 99,224'}
customers     {'customer_id': '99,441 / 99,441', 'customer_unique_id': '96,096 / 99,441'}
products      {'product_id': '32,951 / 32,951'}
sellers       {'seller_id': '3,095 / 3,095'}


## 4. Data dictionary

We print columns and dtypes for every table so we can reason about which ones are identifiers, dates, categoricals, or numerics.

In [5]:
for name, df in raw.items():
    print('=' * 70)
    print(name.upper())
    print(df.dtypes.to_string())
    print()

CUSTOMERS
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix    object
customer_city               object
customer_state              object

ORDERS
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object

ORDER_ITEMS
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64

PRODUCTS
product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm  

## 5. Data quality - missing values

Missing values are expected in a real marketplace: not all orders are approved or delivered, and many reviews have no text. We quantify them per table. **Context matters**: a missing `order_delivered_customer_date` is not an error, it means the order was not delivered.

In [6]:
def missing_report(df):
    # Resumen de valores nulos absolutos y relativos por columna
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        return '    (no missing values)'
    rel = (nulls / len(df) * 100).round(2)
    info = pd.DataFrame({'n': nulls, 'pct': rel})
    return '    ' + '\n    '.join(f"{idx:35s} {row['n']:>8,.0f} ({row['pct']}%)" for idx, row in info.iterrows())

for name, df in raw.items():
    print('=' * 70)
    print(name.upper())
    print(missing_report(df))

CUSTOMERS


    (no missing values)
ORDERS
    order_approved_at                        160 (0.16%)
    order_delivered_carrier_date           1,783 (1.79%)
    order_delivered_customer_date          2,965 (2.98%)
ORDER_ITEMS
    (no missing values)
PRODUCTS
    product_category_name                    610 (1.85%)
    product_name_lenght                      610 (1.85%)
    product_description_lenght               610 (1.85%)
    product_photos_qty                       610 (1.85%)
    product_weight_g                           2 (0.01%)
    product_length_cm                          2 (0.01%)
    product_height_cm                          2 (0.01%)
    product_width_cm                           2 (0.01%)
PAYMENTS
    (no missing values)
REVIEWS


    review_comment_title                  87,656 (88.34%)
    review_comment_message                58,247 (58.7%)
SELLERS
    (no missing values)
GEOLOCATION
    (no missing values)
CATEGORIES
    (no missing values)


> **Key insight:** later we will see that almost all missing values in `orders` are explained by the `order_status` - canceled/unavailable/shipped orders never reached the customer. This is a **structural** null, not a data-entry problem.

## 6. Data quality - duplicates

We look for exact duplicate rows. `geolocation` is the only file that can legitimately repeat zip codes (one zip can have several coordinate points recorded), so exact row duplicates there are suspicious and will be handled in the cleaning step.

In [7]:
for name, df in raw.items():
    print(f"{name:14s} duplicate rows: {df.duplicated().sum():,}")

customers      duplicate rows: 0


orders         duplicate rows: 0
order_items    duplicate rows: 0
products       duplicate rows: 0


payments       duplicate rows: 0
reviews        duplicate rows: 0
sellers        duplicate rows: 0


geolocation    duplicate rows: 261,831
categories     duplicate rows: 0


## 7. Referential integrity

There is no enforced foreign key in the CSVs, so we **verify** that every reference between tables points to an existing row. The expected relationships are:

- `orders.customer_id` -> `customers.customer_id`
- `order_items.order_id` -> `orders.order_id`
- `order_items.product_id` -> `products.product_id`
- `order_items.seller_id` -> `sellers.seller_id`
- `payments.order_id` -> `orders.order_id`
- `reviews.order_id` -> `orders.order_id`
- `products.product_category_name` -> `categories.product_category_name`

In [8]:
def orphan_count(child_df, child_col, parent_df, parent_col):
    # Cuenta claves hijas que no existen en la tabla padre
    return int((~child_df[child_col].isin(parent_df[parent_col])).sum())

checks = [
    ('orders.customer_id',     orders,      'customer_id', customers, 'customer_id'),
    ('order_items.order_id',   order_items, 'order_id',    orders,    'order_id'),
    ('order_items.product_id', order_items, 'product_id',  products,  'product_id'),
    ('order_items.seller_id',  order_items, 'seller_id',   sellers,   'seller_id'),
    ('payments.order_id',      payments,    'order_id',    orders,    'order_id'),
    ('reviews.order_id',       reviews,     'order_id',    orders,    'order_id'),
]
for label, cdf, ccol, pdf, pcol in checks:
    n = orphan_count(cdf, ccol, pdf, pcol)
    print(f'{label:32s} orphans: {n}')

orders.customer_id               orphans: 0
order_items.order_id             orphans: 0
order_items.product_id           orphans: 0
order_items.seller_id            orphans: 0
payments.order_id                orphans: 0
reviews.order_id                 orphans: 0


## 8. Key distributions

We inspect the most relevant categorical variable: `order_status`. This drives every later decision (which orders count as revenue, as delivered, etc.).

In [9]:
status = orders['order_status'].value_counts()
status_df = pd.DataFrame({'count': status, 'percent': (status / status.sum() * 100).round(2)})
status_df

,count,percent
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


> **Interpretation:** ~97% of orders are `delivered`. The remaining statuses describe orders that are still in progress or that never completed (canceled, unavailable), so they are **excluded** from most forward-looking business metrics.

## 9. Time range of the data

We confirm the temporal coverage so downstream time-series plots are interpreted correctly. Note that 2016 only holds a couple of months and 2018 ends in October - these partial years are **not comparable** to full years on absolute totals.

In [10]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

print('purchase date  min:', orders['order_purchase_timestamp'].min())
print('purchase date  max:', orders['order_purchase_timestamp'].max())
print('estimated date max:', orders['order_estimated_delivery_date'].max())
print()
print('Orders per year:')
print(orders['order_purchase_timestamp'].dt.year.value_counts().sort_index())

purchase date  min: 2016-09-04 21:15:19
purchase date  max: 2018-10-17 17:30:18
estimated date max: 2018-11-12 00:00:00

Orders per year:
order_purchase_timestamp
2016      329
2017    45101
2018    54011
Name: count, dtype: int64


## 10. Takeaways

- The dataset spans **Sep 2016 - Oct 2018** (about 99,441 orders, 112,650 order items, 3,095 sellers, 32,951 products).
- The natural grain is the **order** for business analysis and the **order item** for line-level analysis.
- Missing dates in `orders` are **structural** and align with non-delivered statuses; most other files are almost complete.
- `geolocation` is the only file with relevant exact duplicates (to be cleaned next).
- Referential integrity holds: **no orphan keys** across tables (the only mismatch is a few products whose category is not present in the translation table, which we will handle).

Now we are ready to **clean and model the data** in `02_cleaning_preprocessing.ipynb`.